# 방식 B: 모멘텀 종목 선정 + %R 사전 필터

**구조:**
- 매월 첫 거래일: 모멘텀(12-1 & 6-1) 랭킹 산출
- %R(14) > -20 (과매수) 종목은 후보에서 제외
- 남은 종목에서 상위 N개 선정 → 첫 거래일 종가로 체결
- 청산: 쿨다운 만료 + 모멘텀 랭킹 탈락 시 매도 (기존 방식)

**룩어헤드 방지:**
- 모멘텀 신호: 전월 말 확정 종가 (월별 shift)
- %R 필터: **전월 마지막 거래일**의 확정 %R 값 사용
- 체결: 리밸런싱 월 첫 거래일 종가
- 백테스트 루프: 월간

In [ ]:
# ============================================================
# [CELL 1] CONFIG
# ============================================================

CONFIG = {
    # ── 기간 ──
    "START_DATE": "2010-01-01",
    "END_DATE": "2025-01-01",
    "WARMUP_MONTHS": 13,

    # ── 모멘텀 ──
    "N_MOM": 8,
    "MOM_WEIGHT_12": 0.5,
    "MOM_WEIGHT_6": 0.5,

    # ── %R 필터 ──
    "WR_PERIOD": 14,
    "WR_OVERBOUGHT": -20,  # 이 값 이상이면 과매수 → 진입 제외

    # ── 쿨다운 ──
    "COOLDOWN_MONTHS": 3,

    # ── 비용 & 자본 ──
    "COST_RATE": 0.003,
    "INITIAL_CAPITAL": 100_000_000,

    # ── KRX OHLCV CSV (로컬 데이터) ──
    "KRX_CSV": r"C:\Users\jeeho\Desktop\pj3\project3\data\krx\krx_ohlcv_20100101_20261231.csv",

    # ── 데이터 품질 ──
    "MIN_HISTORY_DAYS": 260,
}

print("✅ CONFIG 설정 완료")
for k, v in CONFIG.items():
    if not isinstance(v, list):
        print(f"   {k}: {v}")

In [ ]:
# ============================================================
# [CELL 2] 라이브러리
# ============================================================

import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("fivethirtyeight")

print("✅ 라이브러리 로드 완료")

In [ ]:
# ============================================================
# [CELL 3] KRX CSV 로드 & 종목 티커 추출
# ============================================================

csv_path = CONFIG["KRX_CSV"]

df_krx = pd.read_csv(csv_path)
df_krx["date"] = pd.to_datetime(df_krx["date"])
df_krx["ticker"] = df_krx["ticker"].astype(str).str.zfill(6)

# 기간 필터
df_krx = df_krx[(df_krx["date"] >= CONFIG["START_DATE"]) & (df_krx["date"] <= CONFIG["END_DATE"])]

tickers = sorted(df_krx["ticker"].unique())
print(f"✅ KRX CSV 로드 완료: {csv_path}")
print(f"   총 {len(tickers)}개 종목, {len(df_krx):,}행 데이터")
print(f"   기간: {df_krx['date'].min().strftime('%Y-%m-%d')} ~ {df_krx['date'].max().strftime('%Y-%m-%d')}")

In [ ]:
# ============================================================
# [CELL 4] 데이터 처리: 일간(체결+%R) + 월간(모멘텀+WR_end) (로컬 CSV 기반)
# ============================================================
#
# ★ %R은 일간 H/L/C로 계산하지만, 리밸런싱 시 사용하는 값은
#   "전월 마지막 거래일"의 %R이다. (당월 데이터 사용 금지)
# ★ 모멘텀: 월별 리샘플링 후 shift → 전월 말 확정 종가만 사용
#

stocks_daily = {}    # {ticker: DataFrame[Close, WR]}
stocks_monthly = {}  # {ticker: DataFrame[Close, Momentum, WR_end]}

WR_N = CONFIG["WR_PERIOD"]

for t in tqdm(tickers, desc="데이터 처리",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{percentage:3.0f}%]"):
    sub = df_krx[df_krx["ticker"] == t].set_index("date").sort_index()

    if len(sub) < CONFIG["MIN_HISTORY_DAYS"]:
        continue

    # ── 일간: %R(14) 계산 ──
    highest = sub["high"].rolling(WR_N).max()
    lowest  = sub["low"].rolling(WR_N).min()
    denom = highest - lowest
    sub["WR"] = np.where(denom > 0,
                        (highest - sub["close"]) / denom * (-100),
                        np.nan)

    daily = sub[["close", "WR"]].copy()
    daily.columns = ["Close", "WR"]
    stocks_daily[t] = daily

    # ── 월간: 모멘텀 계산 ──
    monthly = sub["close"].resample("ME").last().dropna().to_frame("Close")
    monthly["Mom_12_1"] = monthly["Close"].shift(1) / monthly["Close"].shift(12) - 1
    monthly["Mom_6_1"]  = monthly["Close"].shift(1) / monthly["Close"].shift(6) - 1
    w12, w6 = CONFIG["MOM_WEIGHT_12"], CONFIG["MOM_WEIGHT_6"]
    monthly["Momentum"] = w12 * monthly["Mom_12_1"] + w6 * monthly["Mom_6_1"]

    # ── 월말 %R 값도 월간 테이블에 저장 (전월 말 기준 필터용) ──
    wr_monthly = sub["WR"].resample("ME").last()
    monthly["WR_end"] = wr_monthly

    monthly = monthly.dropna(subset=["Momentum"])
    if len(monthly) > 0:
        stocks_monthly[t] = monthly

valid_tickers = set(stocks_daily.keys()) & set(stocks_monthly.keys())
stocks_daily   = {t: v for t, v in stocks_daily.items() if t in valid_tickers}
stocks_monthly = {t: v for t, v in stocks_monthly.items() if t in valid_tickers}

print(f"\n분석 가능 종목: {len(valid_tickers)}개")

In [ ]:
# ============================================================
# [CELL 5] 리밸런싱 날짜 생성
# ============================================================

all_dates = sorted(set().union(*(df.index.tolist() for df in stocks_daily.values())))
daily_index = pd.DatetimeIndex(all_dates)

# 매월 첫 거래일
rebal_dates = daily_index.to_series().groupby(
    [daily_index.year, daily_index.month]
).first().values
rebal_dates = pd.DatetimeIndex(sorted(rebal_dates))

# 워밍업 제거
warmup_end = pd.Timestamp(CONFIG["START_DATE"]) + pd.DateOffset(months=CONFIG["WARMUP_MONTHS"])
rebal_dates = rebal_dates[rebal_dates >= warmup_end]

print(f"리밸런싱 횟수: {len(rebal_dates)}회")
print(f"첫 리밸런싱: {rebal_dates[0].strftime('%Y-%m-%d')}")
print(f"마지막: {rebal_dates[-1].strftime('%Y-%m-%d')}")

In [ ]:
# ============================================================
# [CELL 6] 백테스트 엔진 (월간 루프)
# ============================================================
#
# ★ 데이터 흐름:
#   [전월 말] 모멘텀 확정 + %R(14) 확정
#   [당월 첫 거래일] 모멘텀 랭킹에서 %R > -20 종목 제외
#                   → 남은 종목 상위 N개 → 첫 거래일 종가로 체결
#
# ★ 룩어헤드 방지:
#   - 모멘텀: 전월 말 shift(1) 기반
#   - %R: 전월 마지막 거래일의 확정 값
#     → stocks_monthly에 WR_end 컬럼으로 저장됨
#     → rebal_date 이전 최근 월말 WR_end 사용
#   - 체결: 리밸런싱 당일(첫 거래일) 종가
#

def get_momentum(ticker, rebal_date):
    if ticker not in stocks_monthly:
        return None
    df = stocks_monthly[ticker]
    avail = df.loc[:rebal_date - pd.Timedelta(days=1)]
    if avail.empty:
        return None
    return avail["Momentum"].iloc[-1]


def get_wr_last_month(ticker, rebal_date):
    """전월 말 %R 값 반환. 리밸런싱일 이전 가장 최근 월말."""
    if ticker not in stocks_monthly:
        return None
    df = stocks_monthly[ticker]
    avail = df.loc[:rebal_date - pd.Timedelta(days=1)]
    if avail.empty or pd.isna(avail["WR_end"].iloc[-1]):
        return None
    return avail["WR_end"].iloc[-1]


def get_exec_price(ticker, date):
    if ticker not in stocks_daily:
        return None
    df = stocks_daily[ticker]
    future = df.loc[date:]
    if future.empty:
        return None
    return future["Close"].iloc[0]


def run_backtest_B():
    cash = CONFIG["INITIAL_CAPITAL"]
    holdings = {}   # {ticker: {"shares": int, "buy_date": Timestamp}}
    history = []
    trade_log = []
    total_traded = 0.0
    filter_stats = {"total_filtered": 0, "months_with_filter": 0}

    N = CONFIG["N_MOM"]
    COOLDOWN = CONFIG["COOLDOWN_MONTHS"]
    COST = CONFIG["COST_RATE"]
    WR_OB = CONFIG["WR_OVERBOUGHT"]

    for date in rebal_dates:

        # ── (A) 포트폴리오 평가 ──
        port_val = cash
        dead = []
        for t, info in holdings.items():
            p = get_exec_price(t, date)
            if p is not None:
                port_val += info["shares"] * p
            else:
                dead.append(t)
        for t in dead:
            del holdings[t]

        # ── (B) 모멘텀 랭킹 + %R 필터 ──
        candidates = []
        filtered_count = 0
        for t in valid_tickers:
            mom = get_momentum(t, date)
            price = get_exec_price(t, date)
            if mom is None or price is None:
                continue

            # %R 과매수 필터: 전월 말 %R > -20 이면 진입 후보에서 제외
            wr = get_wr_last_month(t, date)
            if wr is not None and wr >= WR_OB:
                filtered_count += 1
                # 단, 이미 보유 중인(쿨다운 보호) 종목은 제외하지 않음
                if t not in holdings:
                    continue

            candidates.append((t, mom))

        if filtered_count > 0:
            filter_stats["total_filtered"] += filtered_count
            filter_stats["months_with_filter"] += 1

        candidates.sort(key=lambda x: x[1], reverse=True)

        # ── (C) 쿨다운 보호 ──
        cooldown_keep = []
        for t, info in holdings.items():
            months_held = ((date.year - info["buy_date"].year) * 12
                           + (date.month - info["buy_date"].month))
            if months_held < COOLDOWN:
                cooldown_keep.append(t)

        # ── (D) 타깃 포트폴리오 ──
        target = cooldown_keep.copy()
        for t, _ in candidates:
            if len(target) >= N:
                break
            if t not in target:
                target.append(t)

        # ── (E) 매도 ──
        for t in list(holdings.keys()):
            if t not in target:
                p = get_exec_price(t, date)
                if p is None:
                    continue
                traded_val = holdings[t]["shares"] * p
                cash += traded_val * (1 - COST)
                total_traded += traded_val
                trade_log.append((date, t, "SELL", holdings[t]["shares"], p))
                del holdings[t]

        # ── (F) 매수/리밸런싱 ──
        if target:
            current_val = cash
            for t in holdings:
                p = get_exec_price(t, date)
                if p:
                    current_val += holdings[t]["shares"] * p

            alloc = current_val / len(target)

            for t in target:
                p = get_exec_price(t, date)
                if p is None or p <= 0:
                    continue

                cur_s = holdings.get(t, {}).get("shares", 0)
                tar_s = int(alloc // p)
                diff = tar_s - cur_s

                if diff > 0:
                    cost = diff * p * (1 + COST)
                    if cash >= cost:
                        cash -= cost
                        total_traded += diff * p
                        old_date = holdings.get(t, {}).get("buy_date", date)
                        holdings[t] = {
                            "shares": tar_s,
                            "buy_date": old_date if t in holdings else date,
                        }
                        trade_log.append((date, t, "BUY", diff, p))
                elif diff < 0:
                    sell_s = abs(diff)
                    traded_val = sell_s * p
                    cash += traded_val * (1 - COST)
                    total_traded += traded_val
                    holdings[t]["shares"] = tar_s
                    trade_log.append((date, t, "TRIM", sell_s, p))

        # ── (G) 기록 ──
        final_val = cash
        for t, info in holdings.items():
            p = get_exec_price(t, date)
            if p:
                final_val += info["shares"] * p
        history.append((date, final_val))

    perf_df = pd.DataFrame(history, columns=["Date", "Value"]).set_index("Date")
    trade_df = pd.DataFrame(trade_log, columns=["Date", "Ticker", "Action", "Shares", "Price"])
    return perf_df, trade_df, total_traded, filter_stats


print("✅ 백테스트 엔진 정의 완료")

In [ ]:
# ============================================================
# [CELL 7] 백테스트 실행
# ============================================================

perf_df, trade_df, total_traded, filter_stats = run_backtest_B()

print(f"백테스트 완료")
print(f"  시작 자산: {CONFIG['INITIAL_CAPITAL']:>20,} 원")
print(f"  최종 자산: {perf_df['Value'].iloc[-1]:>20,.0f} 원")
print(f"  총 매매 건수: {len(trade_df)}건")
print(f"\n%R 필터 효과:")
print(f"  필터 발동 월: {filter_stats['months_with_filter']}회 / {len(rebal_dates)}회")
print(f"  총 제외된 종목 수: {filter_stats['total_filtered']}개")
if filter_stats['months_with_filter'] > 0:
    print(f"  월평균 제외 종목: {filter_stats['total_filtered']/filter_stats['months_with_filter']:.1f}개")

In [ ]:
# ============================================================
# [CELL 8] 성과 분석
# ============================================================

def compute_metrics(df, initial_capital, traded_sum):
    returns = df["Value"].pct_change().dropna()
    years = max((df.index[-1] - df.index[0]).days / 365.25, 0.1)

    final = df["Value"].iloc[-1]
    cagr = ((final / initial_capital) ** (1 / years) - 1) * 100
    cum_ret = (final / initial_capital - 1) * 100

    running_max = df["Value"].cummax()
    mdd = (df["Value"] / running_max - 1).min() * 100

    sharpe = (returns.mean() * 12) / (returns.std() * np.sqrt(12)) if returns.std() > 0 else 0
    turnover = (traded_sum / 2) / df["Value"].mean() / years * 100

    return {
        "누적 수익률": f"{cum_ret:.2f}%",
        "CAGR": f"{cagr:.2f}%",
        "MDD": f"{mdd:.2f}%",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "연간 회전율": f"{turnover:.1f}%",
        "최종 자산": f"{final:,.0f} 원",
    }


metrics = compute_metrics(perf_df, CONFIG["INITIAL_CAPITAL"], total_traded)
print(f"{'='*50}")
print(f"  [방식 B] 전략 성과 요약")
print(f"{'='*50}")
summary_df = pd.DataFrame(list(metrics.items()), columns=["지표", "값"])
display(summary_df.style.hide(axis="index").set_properties(**{"text-align": "left", "padding": "8px"}))

In [ ]:
# ============================================================
# [CELL 9] 연도별 성과
# ============================================================

def yearly_analysis(df, initial_capital):
    yearly = df["Value"].resample("YE").last()
    yearly_ret = yearly.pct_change()
    yearly_ret.iloc[0] = yearly.iloc[0] / initial_capital - 1
    yearly_mdd = df.groupby(df.index.year)["Value"].apply(
        lambda x: (x / x.cummax() - 1).min()
    )
    return pd.DataFrame({
        "연도": yearly_ret.index.year,
        "수익률(%)": (yearly_ret.values * 100).round(2),
        "MDD(%)": (yearly_mdd.values * 100).round(2),
    })

yearly_df = yearly_analysis(perf_df, CONFIG["INITIAL_CAPITAL"])
print(f"{'='*50}")
print(f"  [방식 B] 연도별 성과")
print(f"{'='*50}")
display(yearly_df.style.hide(axis="index")
        .format({"수익률(%)": "{:.2f}", "MDD(%)": "{:.2f}"})
        .set_properties(**{"text-align": "center", "padding": "8px"}))

In [ ]:
# ============================================================
# [CELL 10] 월간 세부 통계
# ============================================================

monthly_ret = perf_df["Value"].pct_change().dropna()
win = monthly_ret[monthly_ret > 0]
lose = monthly_ret[monthly_ret < 0]

detail = {
    "월간 승률": f"{len(win)/len(monthly_ret)*100:.1f}%",
    "평균 상승월": f"{win.mean()*100:.2f}%" if len(win) > 0 else "N/A",
    "평균 하락월": f"{lose.mean()*100:.2f}%" if len(lose) > 0 else "N/A",
    "Profit Factor": f"{abs(win.sum()/lose.sum()):.2f}" if len(lose) > 0 else "N/A",
    "최고 월간": f"{monthly_ret.max()*100:.2f}%",
    "최저 월간": f"{monthly_ret.min()*100:.2f}%",
}

print(f"{'='*50}")
print(f"  [방식 B] 월간 세부 통계")
print(f"{'='*50}")
detail_df = pd.DataFrame(list(detail.items()), columns=["항목", "값"])
display(detail_df.style.hide(axis="index").set_properties(**{"text-align": "left", "padding": "8px"}))

In [ ]:
# ============================================================
# [CELL 11] 시각화
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={"height_ratios": [3, 1]})

ax1 = axes[0]
ax1.plot(perf_df.index, perf_df["Value"], lw=2, color="#FF9800")
ax1.axhline(y=CONFIG["INITIAL_CAPITAL"], color="gray", ls="--", alpha=0.5)
ax1.set_yscale("log")
ax1.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:,.0f}"))
ax1.set_title("[방식 B] 모멘텀 + %R 과매수 필터", fontsize=16, fontweight="bold")
ax1.set_ylabel("자산 (로그 스케일)")
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
dd = (perf_df["Value"] / perf_df["Value"].cummax() - 1) * 100
ax2.fill_between(dd.index, dd.values, 0, color="#F44336", alpha=0.4)
ax2.set_ylabel("낙폭 (%)")
ax2.set_title("Drawdown", fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 룩어헤드 방지 체크리스트

| # | 항목 | 처리 |
|---|------|------|
| 1 | 모멘텀 신호 | 월말 리샘플링 + shift(1) → 전월 말 확정 종가만 사용 |
| 2 | %R 필터 값 | **전월 마지막 거래일**의 확정 %R → `rebal_date - 1일` 이전 데이터 |
| 3 | 체결 가격 | 리밸런싱 월 **첫 거래일 종가** (신호 확정 후 체결) |
| 4 | 보유 종목 보호 | 이미 보유 중인 쿨다운 종목은 %R 필터 적용 안 함 (진입만 필터) |
| 5 | 생존자 편향 | 현재 지수 구성종목 사용 → 편향 존재. 과거 구성종목 데이터로 개선 가능 |